In [1]:
import sys
sys.path.append('/home/azureuser/prathyusha/Kearney/prathyusha')
from utils import *
spark = instantiate_spark_sedona("10g")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/21 13:58:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Session and SedonaContext have been successfully initiated.


In [2]:
from pyspark.sql.functions import *

final_df = spark.read.parquet("abfss://propheus-data-science@propheusdatabay.dfs.core.windows.net/Thailand/thailand_quadkeys/Quadkey_pop_hh_estimated_updated")
final_df_req = final_df.filter(~col('adm1_name').isin('bangkok', 'nonthaburi', 'pathumthani', 'samutprakan', 'samutsakhon', 'nakhonpathom'))
final_df_req.show()

+----------------+------------+--------------------+------------------+-----------------------+------------------+--------------------+---------------+
|         quadkey|   adm1_name|Pop_nso_to_hdx_ratio|   hdx_Quadkey_Pop|nso_adjusted_population|sum_floor_area_sqm|Qk_pop_to_hh_sampled|Qk_estimated_hh|
+----------------+------------+--------------------+------------------+-----------------------+------------------+--------------------+---------------+
|1322121030100010|amnatcharoen|  1.3095653425533569| 142.0465920000001|     186.01929391101703| 76090.12173170949|   2.449373527632673|           76.0|
|1322121021232330|amnatcharoen|  1.3095653425533569|201.32532000000032|      263.6486616504646| 62566.12192884956|   2.488199875170112|          106.0|
|1322121030102232|amnatcharoen|  1.3095653425533569|135.38815800000015|     177.29963950893818| 54976.98828964075|   2.523829708720109|           70.0|
|1322121021312003|amnatcharoen|  1.3095653425533569|199.33478199999976|      261.0419220

In [5]:
final_df_req.filter(col('quadkey').isin(['1322120010111102','1322102232332333'])).show()

+----------------+---------+--------------------+---------------+-----------------------+------------------+--------------------+---------------+
|         quadkey|adm1_name|Pop_nso_to_hdx_ratio|hdx_Quadkey_Pop|nso_adjusted_population|sum_floor_area_sqm|Qk_pop_to_hh_sampled|Qk_estimated_hh|
+----------------+---------+--------------------+---------------+-----------------------+------------------+--------------------+---------------+
|1322120010111102| khonkaen|  1.0225002642448116|      13.488684|     13.792182954314761|340.22871353185116|  3.0703623039123156|            4.0|
|1322102232332333| khonkaen|  1.0225002642448116|6.1779542234763|      6.316959825996866|101.83978518024774|   3.268238428284067|            2.0|
+----------------+---------+--------------------+---------------+-----------------------+------------------+--------------------+---------------+



In [3]:
mapping = spark.read.parquet("abfss://propheus-data-science@propheusdatabay.dfs.core.windows.net/Thailand/thailand_quadkeys/thailand_quadkeys_w_subdistrict")
mapping.show()

+----------------+--------------------+--------------------+-----------------+------------+---------+--------------------+
|         quadkey|    quadkey_geometry|    quadkey_centroid|        adm3_name|   adm2_name|adm1_name|            geometry|
+----------------+--------------------+--------------------+-----------------+------------+---------+--------------------+
|1322122012022123|POLYGON ((101.980...|POINT (101.983337...|Thung Mahacharoen|Wang Nam Yen|  Sa Kaeo|POLYGON ((102.032...|
|1322122012022132|POLYGON ((101.986...|POINT (101.988830...|Thung Mahacharoen|Wang Nam Yen|  Sa Kaeo|POLYGON ((102.032...|
|1322122012022133|POLYGON ((101.991...|POINT (101.994323...|Thung Mahacharoen|Wang Nam Yen|  Sa Kaeo|POLYGON ((102.032...|
|1322122012023022|POLYGON ((101.997...|POINT (101.999816...|Thung Mahacharoen|Wang Nam Yen|  Sa Kaeo|POLYGON ((102.032...|
|1322122012023023|POLYGON ((102.002...|POINT (102.005310...|Thung Mahacharoen|Wang Nam Yen|  Sa Kaeo|POLYGON ((102.032...|
|132212201202302

In [22]:
final_df_joined = final_df_req.select('quadkey', 'nso_adjusted_population', 'Qk_pop_to_hh_sampled').join(
    mapping.select('quadkey', 'adm2_name', 'adm3_name', 'adm1_name'),
    on = 'quadkey',
    how = 'inner'
)
final_df_joined.count()

960076

In [23]:
final_df_joined.show()

+----------------+-----------------------+--------------------+-------------------+------------+------------+
|         quadkey|nso_adjusted_population|Qk_pop_to_hh_sampled|          adm2_name|   adm3_name|   adm1_name|
+----------------+-----------------------+--------------------+-------------------+------------+------------+
|1322010330330112|       3.85783809004375|  3.0752889070291864|         Pang Mapha|   Na Pu Pom|Mae Hong Son|
|1322010331201320|      2.149387830642142|   3.665291787562955|         Pang Mapha|   Na Pu Pom|Mae Hong Son|
|1322010331201323|     3.1295666306991308|   2.997189388366925|         Pang Mapha|   Na Pu Pom|Mae Hong Son|
|1322010331202200|       9.18006211671745|   2.760167183741254|         Pang Mapha|   Na Pu Pom|Mae Hong Son|
|1322010331222012|      3.089849985643525|  3.3434753253878804|         Pang Mapha|   Na Pu Pom|Mae Hong Son|
|1322010331232100|      5.355036234751846|   2.965004625512987|         Pang Mapha|  Pang Mapha|Mae Hong Son|
|132201033

In [24]:
district_level_nso_pop = final_df_joined.groupBy('adm1_name', 'adm2_name').agg(
    sum('nso_adjusted_population').alias('nso_adjusted_hdx_population')
)
district_level_nso_pop.show()

+-------------------+-----------------+---------------------------+
|          adm1_name|        adm2_name|nso_adjusted_hdx_population|
+-------------------+-----------------+---------------------------+
|         Chiang Rai|          Pa Daet|         24578.121527529758|
|                Nan|        Santi Suk|         13083.532156029301|
|     Kamphaeng Phet|  Pang Sila Thong|           27757.4511022764|
|      Maha Sarakham|     Yang Sisurat|          28337.10887818958|
|           Saraburi|         Nong Don|         14834.771568171085|
|   Nong Bua Lam Phu|         Na Klang|          90339.88305616792|
|  Nakhon Ratchasima|      Chum Phuang|          79499.40946119941|
|           Buri Ram|    Non Din Daeng|          19530.01458633106|
|             Roi Et|        Nong Phok|          51058.00226440469|
|   Ubon Ratchathani|      Khong Chiam|         30983.471244773315|
|   Ubon Ratchathani|        Buntharik|          83897.89938579037|
|Nakhon Si Thammarat|Chaloem Phra Kiat|         

In [25]:
district_level_nso_pop.count()

848

In [26]:
district_truth_set = spark.read.csv('/home/azureuser/prathyusha/Kearney/prathyusha/EDA/Population Adjustment - Districts - Final_District_Adjustments.csv', header=True)
district_truth_set = district_truth_set.select('Province', 'District', 'Population - Truth set (nso matched)')
district_truth_set.show()

+-----------------+-------------------+------------------------------------+
|         Province|           District|Population - Truth set (nso matched)|
+-----------------+-------------------+------------------------------------+
|       Udon Thani|          Thung Fon|                         24593.84085|
|Nakhon Ratchasima|              Khong|                         80640.37428|
|         Lop Buri|    Mueang Lop Buri|                         248302.4556|
|        Khon Kaen|          Chum Phae|                         120085.3787|
|      Suphan Buri|            U Thong|                         97047.32322|
|              Nan|             Na Noi|                         31401.04129|
|     Kanchanaburi|          Tha Muang|                         105003.4113|
|      Surat Thani|          Phrasaeng|                         72105.59717|
|          Bangkok|          Lat Phrao|                         246192.3001|
|     Kanchanaburi|           Tha Maka|                         128199.0563|

In [27]:
multiplier_join = district_level_nso_pop.join(
    district_truth_set,
    ((district_level_nso_pop.adm1_name == district_truth_set.Province) &
    (district_level_nso_pop.adm2_name == district_truth_set.District)),
    how = 'inner'
).drop(district_truth_set.Province, district_truth_set.District)

multiplier_join = multiplier_join.withColumn("Multiplier", col('Population - Truth set (nso matched)')/col('nso_adjusted_hdx_population'))
multiplier_join.show()

+-------------------+-----------------+---------------------------+------------------------------------+------------------+
|          adm1_name|        adm2_name|nso_adjusted_hdx_population|Population - Truth set (nso matched)|        Multiplier|
+-------------------+-----------------+---------------------------+------------------------------------+------------------+
|         Chiang Rai|          Pa Daet|         24578.121527529758|                         24026.71372|0.9775650955703783|
|                Nan|        Santi Suk|         13083.532156029301|                         12634.08971|0.9656482331629243|
|     Kamphaeng Phet|  Pang Sila Thong|           27757.4511022764|                         28631.35473|1.0314835690245323|
|      Maha Sarakham|     Yang Sisurat|          28337.10887818958|                         27431.15232|0.9680293228895036|
|           Saraburi|         Nong Don|         14834.771568171085|                         15502.00202|1.0449774672136172|
|   Nong

In [33]:
# multiplier_join_adm1_adm2
multiplier_join.toPandas().to_csv('multiplier_join_adm1_adm2.csv', index=False)

In [28]:
Matchinh_truth_set = multiplier_join.select('adm1_name', 'adm2_name', 'Multiplier').join(
    final_df_joined,
    on = ['adm1_name', 'adm2_name'],
    how = 'inner'
)
Matchinh_truth_set = Matchinh_truth_set.withColumn("Truthset_adjusted_population", col('Multiplier')*col('nso_adjusted_population'))
Matchinh_truth_set = Matchinh_truth_set.withColumn("Truthset_estimated_HH_count", col('Truthset_adjusted_population')/col('Qk_pop_to_hh_sampled'))
Matchinh_truth_set.show()

+---------+------------+------------------+----------------+-----------------------+--------------------+-----------+----------------------------+---------------------------+
|adm1_name|   adm2_name|        Multiplier|         quadkey|nso_adjusted_population|Qk_pop_to_hh_sampled|  adm3_name|Truthset_adjusted_population|Truthset_estimated_HH_count|
+---------+------------+------------------+----------------+-----------------------+--------------------+-----------+----------------------------+---------------------------+
| Buri Ram|Lam Plai Mat|0.8514999328903994|1322120300123002|      2.179658786140838|  3.1149313548904676| Khok Sa-At|           1.855979310122893|         0.5958331335966651|
| Buri Ram|Lam Plai Mat|0.8514999328903994|1322120300123300|     3.2694881792112573|  3.6351255281719204| Khok Sa-At|            2.78396896518434|         0.7658522226010662|
| Buri Ram|Lam Plai Mat|0.8514999328903994|1322120300131003|      4.359317572281676|    3.18131093690353|Mueang Faek|        

In [35]:
Matchinh_truth_set_71 = Matchinh_truth_set.select('adm1_name', 'adm2_name', 'adm3_name', 'quadkey', 'Truthset_estimated_HH_count').withColumnRenamed(
    'Truthset_estimated_HH_count' , 'HH_Count'
)
Matchinh_truth_set_71.show()

+---------+------------+-----------+----------------+------------------+
|adm1_name|   adm2_name|  adm3_name|         quadkey|          HH_Count|
+---------+------------+-----------+----------------+------------------+
| Buri Ram|Lam Plai Mat| Khok Sa-At|1322120300123002|0.5958331335966651|
| Buri Ram|Lam Plai Mat| Khok Sa-At|1322120300123300|0.7658522226010662|
| Buri Ram|Lam Plai Mat|Mueang Faek|1322120300131003|1.1668015776724898|
| Buri Ram|Lam Plai Mat|     Bu Pho|1322120300132201| 35.96388227477043|
| Buri Ram|Lam Plai Mat|     Bu Pho|1322120300132230|1.5317887547621196|
| Buri Ram|Lam Plai Mat|Mueang Faek|1322120300133222|1.6409855152091184|
| Buri Ram|Lam Plai Mat|   Hin Khon|1322120300211211| 2.803688438722988|
| Buri Ram|Lam Plai Mat|Phathai Rin|1322120300223322| 4.897439128893392|
| Buri Ram|Lam Plai Mat|   Hin Khon|1322120300231001|13.638250461343143|
| Buri Ram|Lam Plai Mat|   Hin Khon|1322120300231011|22.756942549606958|
| Buri Ram|Lam Plai Mat|Phathai Rin|132212030023232

In [38]:
Matchinh_truth_set_71.count()

960076

## Combining all Thailand

In [46]:

def to_snake_case(col):
    return lower(
        regexp_replace(col, r'[^a-zA-Z0-9]+', '_')
    )

In [47]:
df_5_provinces = spark.read.parquet('/home/azureuser/prathyusha/Kearney/prathyusha/EDA/selected_5_province_HH_estimates.parquet')
df_5_provinces.show()

+----------------+------------------+---------------+-------------------+------------+
|         quadkey|          HH_Count|      adm3_name|          adm2_name|   adm1_name|
+----------------+------------------+---------------+-------------------+------------+
|1322033103112230|1.2426369739289753|    Laem Fa Pha|   Phra Samut Chedi|Samut Prakan|
|1322033102132031| 8.999603427405397|        Na Khok|Mueang Samut Sakhon|Samut Sakhon|
|1322033103113321| 18.63955460893463|    Laem Fa Pha|   Phra Samut Chedi|Samut Prakan|
|1322033112003121| 39.66109555864982|    Bang Pu Mai|Mueang Samut Prakan|Samut Prakan|
|1322033112012303| 4.807405522260584|    Bang Pu Mai|Mueang Samut Prakan|Samut Prakan|
|1322033112013320| 1.201851380565146|    Bang Pu Mai|Mueang Samut Prakan|Samut Prakan|
|1322033102123301|21.999030600324303|        Na Khok|Mueang Samut Sakhon|Samut Sakhon|
|1322033112003102| 38.45924417808467|    Bang Pu Mai|Mueang Samut Prakan|Samut Prakan|
|1322033102132023|3.9998237455135097|      

In [48]:
df_final = Matchinh_truth_set_71.unionByName(df_5_provinces)

df_final = df_final.withColumn('adm3_name', to_snake_case(col('adm3_name')))
df_final = df_final.withColumn('adm2_name', to_snake_case(col('adm2_name')))
df_final = df_final.withColumn('adm1_name', to_snake_case(col('adm1_name')))
df_final.count()

977366

In [49]:
df_final.show()

+---------+------------+-----------+----------------+------------------+
|adm1_name|   adm2_name|  adm3_name|         quadkey|          HH_Count|
+---------+------------+-----------+----------------+------------------+
| buri_ram|lam_plai_mat| khok_sa_at|1322120300123002|0.5958331335966651|
| buri_ram|lam_plai_mat| khok_sa_at|1322120300123300|0.7658522226010662|
| buri_ram|lam_plai_mat|mueang_faek|1322120300131003|1.1668015776724898|
| buri_ram|lam_plai_mat|     bu_pho|1322120300132201| 35.96388227477043|
| buri_ram|lam_plai_mat|     bu_pho|1322120300132230|1.5317887547621196|
| buri_ram|lam_plai_mat|mueang_faek|1322120300133222|1.6409855152091184|
| buri_ram|lam_plai_mat|   hin_khon|1322120300211211| 2.803688438722988|
| buri_ram|lam_plai_mat|phathai_rin|1322120300223322| 4.897439128893392|
| buri_ram|lam_plai_mat|   hin_khon|1322120300231001|13.638250461343143|
| buri_ram|lam_plai_mat|   hin_khon|1322120300231011|22.756942549606958|
| buri_ram|lam_plai_mat|phathai_rin|132212030023232

In [55]:
bangkok = spark.read.parquet("abfss://propheus-data-science@propheusdatabay.dfs.core.windows.net/Thailand/bangkok/quadkey_level_household_estimates/")
bangkok = bangkok.drop('province_en')
bangkok = bangkok.withColumnRenamed('total_estimated_households_adjusted', 'HH_Count')
bangkok.show()

+---------+----------------+---------+---------+--------+
|adm2_name|         quadkey|adm3_name|adm1_name|HH_Count|
+---------+----------------+---------+---------+--------+
|  bang_na|1322033110221100|  bang_na|  bangkok|    1531|
|  bang_na|1322033110203303|  bang_na|  bangkok|     879|
|  bang_na|1322033110203230|  bang_na|  bangkok|    9671|
|  bang_na|1322033110212223|  bang_na|  bangkok|     698|
|  bang_na|1322033110203222|  bang_na|  bangkok|     956|
|  bang_na|1322033110203212|  bang_na|  bangkok|    1181|
|  bang_na|1322033110203232|  bang_na|  bangkok|    2124|
|  bang_na|1322033110203320|  bang_na|  bangkok|     920|
|  bang_na|1322033110221012|  bang_na|  bangkok|    2263|
|  bang_na|1322033110203323|  bang_na|  bangkok|     658|
|  bang_na|1322033110202333|  bang_na|  bangkok|     353|
|  bang_na|1322033110221000|  bang_na|  bangkok|     613|
|  bang_na|1322033110230002|  bang_na|  bangkok|     679|
|  bang_na|1322033110221013|  bang_na|  bangkok|    2582|
|  bang_na|132

In [56]:
df_final_final = df_final.unionByName(bangkok)
df_final_final.count()

981791

In [57]:
df_final_final.select('quadkey', 'adm3_name', 'adm2_name', 'adm1_name').distinct().count()

977153

In [100]:
df_final_final_final = df_final_final.groupBy('quadkey', 'adm3_name', 'adm2_name', 'adm1_name').agg(sum('HH_Count').alias('HH_Count'))
df_final_final_final = df_final_final_final.withColumn('HH_Count',when(col('quadkey') == '1322033110000021',10527).otherwise(col('HH_Count')))
df_final_final_final = df_final_final_final.withColumn('HH_Count',when(col('quadkey') == '1322033110320302',103).otherwise(col('HH_Count')))
df_final_final_final.count()

981418

In [101]:
df_final_final_final.select('quadkey', 'adm3_name', 'adm2_name', 'adm1_name').distinct().count()

981418

In [102]:
df_final_final_final = df_final_final_final.filter(col('HH_Count') > 0)
df_final_final_final = df_final_final_final.withColumn('HH_Count', ceil(col('HH_Count')))
df_final_final_final.toPandas().describe()

,HH_Count
count,981037.000000
mean,26.989265
std,111.061506
min,1.000000
25%,2.000000
50%,5.000000
75%,21.000000
max,11787.000000


In [103]:
df_final_final_final.count()

981037

In [104]:
df_final_final_final.toPandas().to_csv('Thailand_HH_Count.csv', index=False)

In [106]:
df_final_final_final.select(sum('HH_Count')).show()

+-------------+
|sum(HH_Count)|
+-------------+
|     26477468|
+-------------+



In [91]:
df_final_final_final.toPandas()['HH_Count'].describe()

count    981418.000000
mean         26.489837
std         111.054433
min           0.000000
25%           1.520495
50%           4.466675
75%          20.509905
max       11787.000000
Name: HH_Count, dtype: float64

In [92]:
df_final_final_final.select('adm1_name','adm2_name','adm3_name').distinct().count()

7417

In [93]:
df_final_final_final.select(sum('HH_Count')).show()

+--------------------+
|       sum(HH_Count)|
+--------------------+
|2.5997603172088426E7|
+--------------------+



In [94]:
df_final_final_final.toPandas().isna().sum()

quadkey      0
adm3_name    0
adm2_name    0
adm1_name    0
HH_Count     0
dtype: int64

In [61]:
df_final_final_final.show()

+----------------+--------------+--------------+------------+------------------+
|         quadkey|     adm3_name|     adm2_name|   adm1_name|          HH_Count|
+----------------+--------------+--------------+------------+------------------+
|1322120300312333|    thamenchai|  lam_plai_mat|    buri_ram|10.571285846115003|
|1322120300323213| nong_bua_khok|  lam_plai_mat|    buri_ram|0.5925368108237239|
|1322120300300232|      khok_lam|  lam_plai_mat|    buri_ram|0.6238949941687498|
|1322120300132231|        bu_pho|  lam_plai_mat|    buri_ram|1.1184703548448311|
|1322120300301130|     talat_pho|  lam_plai_mat|    buri_ram|0.8338654743656927|
|1322120300313032|    thamenchai|  lam_plai_mat|    buri_ram| 83.55429360407037|
|1322120300123012|    khok_sa_at|  lam_plai_mat|    buri_ram|0.7891076394048853|
|1322120300121313|    khok_sa_at|  lam_plai_mat|    buri_ram|3.9898632074513287|
|1322120302000313|   phathai_rin|  lam_plai_mat|    buri_ram|2.0713947517102063|
|1322120300231132|  lam_plai

In [74]:
df_final_final_final.select('quadkey').distinct().count()

981418

In [ ]:
df_final_final_final = df_final_final_final.filter(col('HH_Count') > 0)
df_final_final_final = df_final_final_final.withColumn('HH_Count', col('HH_Count').cast('integer'))
df_final_final_final = df_final_final_final.filter(col('HH_Count') > 0)
df_final_final_final.toPandas().describe()

,HH_Count
count,845295.000000
mean,30.193314
std,121.287899
min,1.000000
25%,2.000000
50%,6.000000
75%,25.000000
max,19844.000000


In [78]:
df_final_final_final.orderBy(col('HH_Count').desc()).show()

+----------------+-------------------+-------------------+------------+--------+
|         quadkey|          adm3_name|          adm2_name|   adm1_name|HH_Count|
+----------------+-------------------+-------------------+------------+--------+
|1322033110000021|            ban_mai|           pak_kret|  nonthaburi|   19844|
|1322033110320302|       bang_chalong|          bang_phli|samut_prakan|   12345|
|1322033110022233|        huai_khwang|        huai_khwang|     bangkok|   11787|
|1322033110201222|       phra_khanong|        khlong_toei|     bangkok|   10176|
|1322033101310233|        dao_khanong|          thon_buri|     bangkok|    9891|
|1322033110000001|            ban_mai|           pak_kret|  nonthaburi|    9867|
|1322033110203230|            bang_na|            bang_na|     bangkok|    9671|
|1322031332201302|       khlong_nueng|       khlong_luang|pathum_thani|    9075|
|1322033101310231|         talat_phlu|          thon_buri|     bangkok|    8894|
|1322031323322213|     bang_

In [80]:
from utils import *

In [82]:
@udf(StringType())
def quadkey_to_wkt_geometry(qk):
    import mercantile
    from shapely.geometry import shape
    if not qk:
        return None
    try:

        tile_feature = mercantile.feature(mercantile.quadkey_to_tile(qk))
        return shape(tile_feature['geometry']).wkt
    except:
        return None

In [83]:
df_final_final_final = df_final_final_final.withColumn('quadkey_geometry', quadkey_to_wkt_geometry('quadkey'))
df_final_final_final.show()

+----------------+--------------+--------------+------------+--------+--------------------+
|         quadkey|     adm3_name|     adm2_name|   adm1_name|HH_Count|    quadkey_geometry|
+----------------+--------------+--------------+------------+--------+--------------------+
|1322120300312333|    thamenchai|  lam_plai_mat|    buri_ram|      10|POLYGON ((102.958...|
|1322120300132231|        bu_pho|  lam_plai_mat|    buri_ram|       1|POLYGON ((102.936...|
|1322120300313032|    thamenchai|  lam_plai_mat|    buri_ram|      83|POLYGON ((102.974...|
|1322120300121313|    khok_sa_at|  lam_plai_mat|    buri_ram|       3|POLYGON ((102.914...|
|1322120302000313|   phathai_rin|  lam_plai_mat|    buri_ram|       2|POLYGON ((102.694...|
|1322120300231132|  lam_plai_mat|  lam_plai_mat|    buri_ram|      55|POLYGON ((102.821...|
|1322120300310312|   mueang_faek|  lam_plai_mat|    buri_ram|       1|POLYGON ((102.952...|
|1322120302101003|    khok_klang|  lam_plai_mat|    buri_ram|      56|POLYGON ((

In [84]:
df_final_final_final.orderBy(col('HH_Count').desc()).show()

+----------------+-------------------+-------------------+------------+--------+--------------------+
|         quadkey|          adm3_name|          adm2_name|   adm1_name|HH_Count|    quadkey_geometry|
+----------------+-------------------+-------------------+------------+--------+--------------------+
|1322033110000021|            ban_mai|           pak_kret|  nonthaburi|   19844|POLYGON ((100.552...|
|1322033110320302|       bang_chalong|          bang_phli|samut_prakan|   12345|POLYGON ((100.744...|
|1322033110022233|        huai_khwang|        huai_khwang|     bangkok|   11787|POLYGON ((100.563...|
|1322033110201222|       phra_khanong|        khlong_toei|     bangkok|   10176|POLYGON ((100.590...|
|1322033101310233|        dao_khanong|          thon_buri|     bangkok|    9891|POLYGON ((100.475...|
|1322033110000001|            ban_mai|           pak_kret|  nonthaburi|    9867|POLYGON ((100.552...|
|1322033110203230|            bang_na|            bang_na|     bangkok|    9671|PO

In [ ]:
df_final_final_final.filter(col('HH_Count'))

## Validations

In [11]:
District_HH_count = Matchinh_truth_set.groupBy('adm1_name', 'adm2_name').agg(
    sum('Truthset_estimated_HH_count').alias('Truthset_estimated_HH_count')
)
District_HH_count.show()

+-------------------+-----------------+---------------------------+
|          adm1_name|        adm2_name|Truthset_estimated_HH_count|
+-------------------+-----------------+---------------------------+
|          Bueng Kan|     Phon Charoen|          12457.64846471362|
|           Buri Ram|          Krasang|         24955.945337244244|
|           Buri Ram|     Lam Plai Mat|         32837.496833696394|
|           Buri Ram|    Non Din Daeng|          8923.438505503214|
|       Chachoengsao|   Bang Nam Priao|         38398.099432736635|
|       Chachoengsao|       Plaeng Yao|         23142.965758187805|
|         Chiang Rai|        Doi Luang|          7229.027072022763|
|         Chiang Rai|     Mae Fa Luang|          27526.35061722876|
|         Chiang Rai|          Pa Daet|          9195.349528808569|
|            Kalasin|         Don Chan|          5217.434431561696|
|            Kalasin|       Yang Talat|          37326.48097030411|
|     Kamphaeng Phet|  Pang Sila Thong|         

In [12]:
estm_province_hh = District_HH_count.groupBy('adm1_name').agg(sum('Truthset_estimated_HH_count').alias('Truthset_estimated_HH_count'))
estm_province_hh.show()

+-------------------+---------------------------+
|          adm1_name|Truthset_estimated_HH_count|
+-------------------+---------------------------+
|       Chachoengsao|          354958.4279098904|
|        Phitsanulok|         381686.96170950506|
|         Narathiwat|          197925.3528553552|
|         Phetchabun|          335624.7126000605|
|     Kamphaeng Phet|          287030.1425312369|
|      Maha Sarakham|          309884.0630748085|
|                Tak|         189426.87162253424|
|Nakhon Si Thammarat|         523908.58598581306|
|        Phatthalung|         175160.33220115537|
|          Khon Kaen|           678663.371119605|
|             Roi Et|         367345.89397383056|
|           Lop Buri|         307311.81893243134|
|         Chiang Rai|          430866.6011265144|
|               Trat|         110040.22852363532|
|          Sukhothai|         244251.83209032303|
|        Surat Thani|           429399.212832271|
|   Ubon Ratchathani|          605607.6498995676|


In [14]:
nso_hh_count = spark.read.csv('/home/azureuser/prathyusha/Kearney/prathyusha/EDA/pop_hhcount_nso_w_geom.csv', header=True).select(
    'Province', '2022_HH_count'
)
nso_hh_count.show()

+--------------------+-------------+
|            Province|2022_HH_count|
+--------------------+-------------+
|             bangkok|      3369140|
|         samutprakan|       924779|
|          nonthaburi|       608941|
|         pathumthani|       675573|
|phranakhonsiayutt...|       311101|
|            angthong|        82287|
|             lopburi|       266958|
|            singburi|        62131|
|             chainat|       101490|
|            saraburi|       253998|
|            chonburi|       690875|
|              rayong|       401393|
|         chanthaburi|       190627|
|                trat|        97383|
|        chachoengsao|       318295|
|         prachinburi|       249575|
|         nakhonnayok|        88356|
|              sakaeo|       212976|
|          ratchaburi|       250405|
|        kanchanaburi|       260639|
+--------------------+-------------+
only showing top 20 rows



In [15]:
final_province_check = estm_province_hh.join(
    nso_hh_count,
    regexp_replace(lower(estm_province_hh.adm1_name), " ", "") == nso_hh_count.Province,
    how = 'inner'
)
final_province_check.count()

71

In [ ]:
final_province_check_pd = final_province_check.toPandas()


,adm1_name,Truthset_estimated_HH_count,Province,2022_HH_count
0,Chachoengsao,354958.427910,chachoengsao,318295
1,Phitsanulok,381686.961710,phitsanulok,323784
2,Narathiwat,197925.352855,narathiwat,181888
3,Phetchabun,335624.712600,phetchabun,292360
4,Kamphaeng Phet,287030.142531,kamphaengphet,253396
...,...,...,...,...
66,Mukdahan,150510.762869,mukdahan,132432
67,Mae Hong Son,95524.941337,maehongson,81739
68,Nakhon Phanom,200542.783051,nakhonphanom,173917
69,Krabi,147395.817389,krabi,132475


In [18]:
final_province_check_pd.dtypes

adm1_name                       object
Truthset_estimated_HH_count    float64
2022_HH_count                   object
dtype: object

In [20]:
# final_province_check_pd = final_province_check_pd.drop(columns={'Province'})
final_province_check_pd['2022_HH_count'] = final_province_check_pd['2022_HH_count'].astype(int)
final_province_check_pd['%_diff'] = (final_province_check_pd['Truthset_estimated_HH_count'] - final_province_check_pd['2022_HH_count']) * 100/final_province_check_pd['2022_HH_count']
final_province_check_pd

,adm1_name,Truthset_estimated_HH_count,2022_HH_count,%_diff
0,Chachoengsao,354958.427910,318295,11.518694
1,Phitsanulok,381686.961710,323784,17.883207
2,Narathiwat,197925.352855,181888,8.817158
3,Phetchabun,335624.712600,292360,14.798438
4,Kamphaeng Phet,287030.142531,253396,13.273352
...,...,...,...,...
66,Mukdahan,150510.762869,132432,13.651355
67,Mae Hong Son,95524.941337,81739,16.865806
68,Nakhon Phanom,200542.783051,173917,15.309477
69,Krabi,147395.817389,132475,11.263119


In [21]:
final_province_check_pd['%_diff'].describe()

count    71.000000
mean     13.599212
std       2.592892
min       7.392183
25%      11.844447
50%      13.563154
75%      14.902295
max      21.086709
Name: %_diff, dtype: float64